<div style="border-top:4px solid #0f766e;padding:28px 0 18px">
<div style="color:#0f766e;font-size:13px;font-weight:700">LAB 01 · DATA WAREHOUSING WITH APACHE DORIS</div>
<h1>连接 Doris，查询 WWI 历史订单</h1>
<p>先从十笔可核对的历史订单认识 Doris，再在 D05 批量导入完整业务表。</p>
<p>约 25 分钟 · WWI 官方样例的转换子集 · 目标 Doris 4.1.3</p></div>

[讲义](course1_introduction_to_apache_doris.md) · [测验](quiz1_doris_fundamentals.ipynb) · [课程目录](../README.md)

完成后得到十笔订单、税前订单金额 12220.60 和两天的汇总。本节数据随仓库提供，不需要 S3 密钥。
来源为 Microsoft Wide World Importers（MIT），本来就是模拟批发业务，不是真实企业交易。详见[数据说明](../../datasets/README.md)。


### 初始化实验工具

下一格只加载 Python 工具和显示样式，不连接数据库、不启动 Docker、不写入数据。
出现“实验工具已就绪”后再继续。重启内核后，需要从这里重新执行。

如果出现 ModuleNotFoundError，先按[环境说明](../../environments/single-node/README.md)
安装依赖，并确认当前 Notebook 使用的是安装这些依赖的 Python 环境。


In [ ]:
from pathlib import Path
import os
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "dw_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course import WarehouseLab
from dw_course.runtime import expect, fixture
from dw_course.wwi import HISTORY_COLUMNS, history_rows, sample
from dw_course.docker_runtime import prepare_environment
from dw_course.ui import card, install_styles

install_styles()
card("尚未连接数据库或启动容器。", "ok", "实验工具已就绪")


## 1. 选择实验环境

选择一种方式，不要同时配置两套环境：

| 方式 | 配置方法 | 会发生什么 |
| --- | --- | --- |
| 讲师已提供 Doris | 保持 USE_DOCKER 为 False，填写 FE_HOST / FE_PORT | 只连接已有实例，不管理服务器进程 |
| 自己准备 Docker 沙箱 | 把 USE_DOCKER 改为 True，先启动 Docker Desktop / Engine | 启动课程 02 独立沙箱，使用 FE 端口 52030 |

配置格优先沿用讲师在启动 Jupyter 时设置的 DW_* 参数。没有预置时，默认连接本机 9030。
**127.0.0.1 是运行 Jupyter 内核的机器，不一定是打开浏览器的 Mac。**

本 Lab 会创建指定的实验库，并在第 4 步删除、重建其中的 **orders_sample**。
确认这张表是可重置的教学数据后，把 ALLOW_LAB_WRITES 改为 True。
其他表不在本节重置范围内；不要填写生产实例或他人的实验库。

实验库名必须以 dw_course_l1_ 开头。多人共用实例时可各加自己的后缀。
密码从进程环境 DW_PASSWORD 读取；需要交互输入时取消 getpass 行的注释，不要把密码写入文件。


In [ ]:
USE_DOCKER = os.environ.get("DW_START_SANDBOX", "no") == "yes"
ALLOW_LAB_WRITES = os.environ.get("DW_ALLOW_WRITES", "no") == "yes"

FE_HOST = os.environ.get("DW_HOST", "127.0.0.1")
FE_PORT = int(os.environ.get("DW_PORT", "9030"))
DATABASE = os.environ.get("DW_DATABASE", "dw_course_l1_demo")
USER = os.environ.get("DW_USER", "root")
# from getpass import getpass
# os.environ["DW_PASSWORD"] = getpass("Doris password: ")

print("环境方式：", "课程 Docker 沙箱" if USE_DOCKER else "已有 Doris")
print("实验数据库：", DATABASE)
print("允许重建本节表：", ALLOW_LAB_WRITES)


### 启动（可选）并连接

检查上一格输出，再运行下一格。没有确认写入时会停止，不会启动容器。

Docker 模式会校验课程 Compose 文件、启动沙箱、等待健康检查通过并测试 SELECT 1，
然后配置连接地址。首次拉取镜像可能需要数分钟。
已有实例模式不会启动或重启 Doris。

WarehouseLab 会连接 FE，创建并选择刚才指定的实验数据库，
设置本会话时区为 +08:00。本节先关闭会话 Group Commit，便于逐次观察写入结果。


In [ ]:
if not ALLOW_LAB_WRITES:
    raise RuntimeError("请先确认 orders_sample 可以重建，再将 ALLOW_LAB_WRITES 设为 True。")

from dw_course.runtime import identifier
identifier(DATABASE)
if not DATABASE.startswith("dw_course_l1_"):
    raise ValueError("请使用 dw_course_l1_ 开头的独立实验库。")

os.environ.update({
    "DW_HOST": FE_HOST,
    "DW_PORT": str(FE_PORT),
    "DW_DATABASE": DATABASE,
    "DW_USER": USER,
    "DW_ALLOW_WRITES": "yes",
    "DW_START_SANDBOX": "yes" if USE_DOCKER else "no",
})
if USE_DOCKER:
    prepare_environment()

lab = WarehouseLab()
lab.sql("SELECT 1 AS connection_ok, DATABASE() AS current_database", title="连接与当前数据库");


**预期结果：** connection_ok 为 1，current_database 等于你指定的实验库。

**遇到问题：**

- Connection refused：核对 FE 查询端口及服务是否已启动，不要误用 FE HTTP 端口。
- Access denied：确认账号、密码以及创建实验库和表所需权限。
- Docker 启动失败：确认 Docker 正在运行、镜像可下载且端口没有冲突。详见[环境排查](../../environments/single-node/README.md)。
- 不要通过停止他人的服务、删除数据库或 Docker 数据卷来解决连接失败。


## 2. 检查 FE 与 BE

FE 接收 SQL 并规划查询；BE 执行查询，在本节的存算一体环境中也保存内部表数据。
先确认节点存活，再开始写入。


In [ ]:
lab.sql("SELECT VERSION() AS protocol_version, @@version_comment AS version_comment", title="连接版本信息");
lab.sql("SHOW FRONTENDS", title="FE 节点");
lab.sql("SHOW BACKENDS", title="BE 节点");


**预期结果：** 课程单节点沙箱有一个存活 FE 和一个存活 BE，Alive 为 true。
讲师提供的集群可能有更多节点。记录 FE/BE 的 Version 字段；如果是开发构建，
不要把它当成正式 4.1.3。SELECT VERSION() 可能返回协议兼容版本。

如果 BE 不存活，先处理环境问题，不要继续建表、写入来测试是否“碰巧能成功”。


## 3. 理解 WWI 订单样本

从 WWI Orders 选择 2013-01-01 与 2013-01-02 各五笔订单，并按订单汇总 OrderLines。
这里只为入门准备一单一行的投影；D05 仍保留原始多表结构，不把它当作必须拼宽表的要求。

| 字段 | 含义 |
| --- | --- |
| order_id、customer_id | 保留 WWI 原始业务标识 |
| order_date | 源订单日期 |
| order_amount | 明细 Quantity × UnitPrice 合计，未含税，不是收款 |
| line_count | 源订单对应的商品明细行数 |
| data_source | WWI；与后续 COURSE_SIMULATION 新订单区分 |

十笔税前订单金额合计 12220.60。样本没有逐订单支付字段，不能推断“未支付”或“已收到全部货款”。
数据文件：[wwi/sample.json](../../datasets/wwi/sample.json)；选择的订单号为 1–5、80–84。


## 4. 创建第一张内部表

下面展示完整 DDL：DECIMAL 保存金额，DATE 保存原订单日期。
DUPLICATE KEY(order_id) 用于排序，不强制唯一；单桶、单副本用于本节小样本。

**重置提示：** 下一格只删除当前实验库中的 orders_sample，不删除整个库。


In [ ]:
lab.execute("DROP TABLE IF EXISTS orders_sample")
lab.execute("""
CREATE TABLE orders_sample (
    order_id BIGINT NOT NULL,
    customer_id BIGINT NOT NULL,
    order_date DATE NOT NULL,
    order_amount DECIMAL(18,2) NOT NULL,
    line_count INT NOT NULL,
    data_source VARCHAR(32) NOT NULL
)
DUPLICATE KEY(order_id)
DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES ("replication_num"="1")
""")
lab.sql("DESC orders_sample", title="历史订单投影字段");
lab.sql("SHOW CREATE TABLE orders_sample", title="实际建表定义");


**预期结果：** 六个字段，Duplicate Key、单桶、单副本。表已创建，但数据尚未写入。


## 5. 写入十笔历史订单

显式 INSERT 便于看清字段；D05 再通过文件导入全部业务表。数据来自 WWI，金额由原始明细汇总。
不要单独反复执行本格：Duplicate Key 会追加；完整重跑先执行第 4 步。


In [ ]:
lab.execute("""
INSERT INTO orders_sample (
    order_id, customer_id, order_date, order_amount, line_count, data_source
) VALUES
(1, 832, '2013-01-01', 2300.00, 1, 'WWI'),
(2, 803, '2013-01-01', 405.00, 2, 'WWI'),
(3, 105, '2013-01-01', 90.00, 1, 'WWI'),
(4, 57, '2013-01-01', 445.20, 3, 'WWI'),
(5, 905, '2013-01-01', 704.00, 3, 'WWI'),
(80, 543, '2013-01-02', 1138.00, 3, 'WWI'),
(81, 456, '2013-01-02', 376.00, 2, 'WWI'),
(82, 487, '2013-01-02', 234.00, 2, 'WWI'),
(83, 154, '2013-01-02', 6220.40, 4, 'WWI'),
(84, 111, '2013-01-02', 308.00, 2, 'WWI')
""")
lab.sql("SELECT * FROM orders_sample ORDER BY order_id", title="核对 WWI 十笔订单");


**预期结果：** 订单号 1–5、80–84，共十行；订单 1 的税前金额为 2300.00。出现二十行时，检查是否重复执行 INSERT，再按声明范围重建。


## 6. 核对总量，也核对明细

COUNT(*) 是投影行数；一单一行时才等于订单数。若直接数 OrderLines，得到的是商品明细数。
逐字段对照仓库里的固定样本，避免总额偶然相同掩盖错误。


In [ ]:
lab.sql("SELECT COUNT(*) AS order_count, SUM(order_amount) AS order_amount FROM orders_sample", title="税前订单金额")
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_sample"), [(10, "12220.60")])
expect(lab.query("""
SELECT order_id, customer_id, CAST(order_date AS STRING), order_amount, line_count, data_source
FROM orders_sample ORDER BY order_id
"""), history_rows())


**预期结果：** 十行、12220.60，所有字段匹配。这个结果只表示税前订单金额，不代表收款、含税发票金额或利润。


## 7. 回答业务问题：每天的订单情况

按原始订单日期分组，固定排序后核对结果。这个十单子集不是 WWI 两天的全部订单，不把子集汇总当作整天经营业绩。


In [ ]:
daily_sql = """
SELECT CAST(order_date AS STRING) AS order_date, COUNT(*) AS order_count,
       SUM(order_amount) AS order_amount
FROM orders_sample GROUP BY order_date ORDER BY order_date
"""
lab.sql(daily_sql, title="样本订单的每日汇总")
expect(lab.query(daily_sql), [("2013-01-01", 5, "3944.20"), ("2013-01-02", 5, "8276.40")])


**预期结果：** 2013-01-01 五单、3944.20；2013-01-02 五单、8276.40。只描述这十笔订单，不据此推断完整销售趋势。


## 8. 自己动手：找出税前金额至少 1000.00 的订单

先写 SQL 返回 order_id、order_date、order_amount，再按订单号排序。
预期三笔，金额合计 9658.40。下面是参考答案，建议先自己尝试。


In [ ]:
lab.sql("""
SELECT order_id, order_date, order_amount FROM orders_sample
WHERE order_amount >= 1000.00 ORDER BY order_id
""", title="金额筛选：参考答案")
expect(lab.query("SELECT order_id FROM orders_sample WHERE order_amount >= 1000.00 ORDER BY order_id"),
       [(1,), (80,), (83,)])
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_sample WHERE order_amount >= 1000.00"),
       [(3, "9658.40")])


## Lab 完成

你已完成连接、检查节点、显式建表、写入 WWI 子集、核对和每日分析。
请解释：为什么订单金额不等于收款？为什么 ORDER_LINES 的行数不等于订单数？为什么重复 INSERT 会翻倍？

继续 [Quiz 1](quiz1_doris_fundamentals.ipynb)，然后进入 [D02](../module02-architecture/course2_doris_architecture.md)。
重跑从初始化开始，仅重建 orders_sample。课程数据保留在实验库；关闭内核释放连接。
本节没有验证持续接入、完整历史规模或生产性能。
